# 24. Isolation Forest and anomaly detection

**Level:** Intermediate

**Prerequisites:** 11-12: splits and preprocessing.

[Course index](00_START_HERE.ipynb) | [Previous](23_PCA_and_Dimensionality_Reduction.ipynb) | [Next](25_Reinforcement_Learning_and_Bandits.ipynb)

**Topics:** Isolation Forest, Anomaly detection, Novelty detection, Contamination, Drift.

**Matching script:** `examples\24_Isolation_Forest_and_Anomaly_Detection.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

An anomaly is unusual relative to a reference distribution; it is not automatically an outage,
attack, or policy violation. Rare valid traffic can look anomalous, while a common failure can
blend into a corrupted baseline. Domain interpretation and evaluation labels remain important.

Isolation Forest repeatedly partitions data with randomized feature thresholds. Isolated points
tend to have shorter path lengths. scikit-learn's predict returns 1 for inliers and -1 for
outliers; score direction and threshold conventions must be checked rather than guessed.

Contamination configures a threshold policy, not the known truth about deployment. Fit a
reference on appropriate historical data, then score future observations. Monitor changes in
feature distributions and alert rates, and evaluate precision/recall when incident labels exist.
Do not silently retrain on anomalous inputs without a reviewed update policy.

## Mathematics or data contract

The detector's score ranks unusualness under its learned reference; it is not a calibrated incident probability.

## Important limitation

The distant synthetic example is deliberately easy and does not establish operational false-alarm rates.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Fit a reference and score new observations

**Purpose:** Use explicit inlier/outlier semantics.

### Read the code in this order
1. Generate a compact historical reference distribution.
2. Fit a bounded Isolation Forest.
3. Score a typical point and a distant new observation.
4. Check both the score ordering and labels.

### Syntax and behavior to notice
- `fit(...)` learns model or preprocessing state from the supplied training data.
- `predict(...)` returns the estimator's chosen output, not a guarantee of correctness.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
decision scores: [0.181, -0.172]
inlier/outlier labels: [1, -1]
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np
from sklearn.ensemble import IsolationForest

rng = np.random.default_rng(42)
historical = rng.normal(0, 0.3, (200, 2))
detector = IsolationForest(n_estimators=50, contamination=0.05, random_state=42, n_jobs=1)
detector.fit(historical)
new = np.array([[0., 0.], [8., 8.]])
scores = detector.decision_function(new)
labels = detector.predict(new)
assert scores[1] < scores[0] and labels.tolist() == [1, -1]
print("decision scores:", np.round(scores, 3).tolist())
print("inlier/outlier labels:", labels.tolist())

decision scores: [0.181, -0.172]
inlier/outlier labels: [1, -1]


### Interpret example 1

The result should demonstrate this purpose: **Use explicit inlier/outlier semantics.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Generate subtle anomalies, choose thresholds on validation data, and evaluate missed events and alert volume.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Isolation Forest` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: The detector's score ranks unusualness under its learned reference; it is not a calibrated incident probability.
- Name the boundary of the demonstration: The distant synthetic example is deliberately easy and does not establish operational false-alarm rates.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.